# 📊 Industry Folio Count — Data Cleaning

## Objective
Clean and validate the industry folio count dataset to prepare it for analysis in the Mutual Fund Analytics project.

## Source File
`data/raw/06_industry_folio_count.csv`

## Output File
`data/processed/cleaned_industry_folio_count.csv`

## Cleaning Workflow
1. Load the raw dataset and inspect its structure.
2. Check column names, data types, and missing values.
3. Identify duplicate records.
4. Validate dates, folio counts, and other numeric fields.
5. Apply appropriate cleaning operations without changing valid source data.
6. Export and verify the cleaned dataset.

In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
# Locate the project folder
project_path = Path.cwd().parent

# Load the raw dataset
folio_file = (
    project_path
    / "data"
    / "raw"
    / "06_industry_folio_count.csv"
)

folio_df = pd.read_csv(folio_file)

print("Dataset loaded successfully!")
print("Shape:", folio_df.shape)

print("\nColumn names:")
print(folio_df.columns.tolist())

print("\nData types:")
print(folio_df.dtypes)

print("\nFirst 5 rows:")
display(folio_df.head())

Dataset loaded successfully!
Shape: (21, 6)

Column names:
['month', 'total_folios_crore', 'equity_folios_crore', 'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore']

Data types:
month                      str
total_folios_crore     float64
equity_folios_crore    float64
debt_folios_crore      float64
hybrid_folios_crore    float64
others_folios_crore    float64
dtype: object

First 5 rows:


,month,total_folios_crore,equity_folios_crore,debt_folios_crore,hybrid_folios_crore,others_folios_crore
0,2022-01,13.26,9.28,1.86,0.80,1.33
1,2022-04,13.91,9.74,1.95,0.83,1.39
2,2022-07,13.85,9.69,1.94,0.83,1.38
3,2022-10,14.12,9.88,1.98,0.85,1.41
4,2023-01,14.81,10.37,2.07,0.89,1.48


In [3]:
import pandas as pd
from pathlib import Path

project_path = Path.cwd().parent

folio_file = (
    project_path
    / "data"
    / "raw"
    / "06_industry_folio_count.csv"
)

folio_df = pd.read_csv(folio_file)

print("Dataset loaded successfully!")
print("Shape:", folio_df.shape)

Dataset loaded successfully!
Shape: (21, 6)


In [4]:
# Check missing values
print("Missing values:")
print(folio_df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", folio_df.duplicated().sum())

# Check duplicate months
print(
    "Duplicate months:",
    folio_df["month"].duplicated().sum()
)

# Check date format
folio_dates = pd.to_datetime(
    folio_df["month"],
    format="%Y-%m",
    errors="coerce"
)

print("\nInvalid months:", folio_dates.isnull().sum())
print("Earliest month:", folio_dates.min())
print("Latest month:", folio_dates.max())

# Check numeric values
numeric_cols = folio_df.select_dtypes(include="number").columns

print("\nNumeric summary:")
display(folio_df[numeric_cols].describe().T)

# Check whether total folios approximately equal category totals
category_total = (
    folio_df["equity_folios_crore"]
    + folio_df["debt_folios_crore"]
    + folio_df["hybrid_folios_crore"]
    + folio_df["others_folios_crore"]
)

print(
    "\nRows where total differs from category sum:",
    (~(folio_df["total_folios_crore"].round(2)
       == category_total.round(2))).sum()
)

Missing values:
month                  0
total_folios_crore     0
equity_folios_crore    0
debt_folios_crore      0
hybrid_folios_crore    0
others_folios_crore    0
dtype: int64

Duplicate rows: 0
Duplicate months: 0

Invalid months: 0
Earliest month: 2022-01-01 00:00:00
Latest month: 2025-12-01 00:00:00

Numeric summary:


,count,mean,std,min,25%,50%,75%,max
total_folios_crore,21.0,19.829524,4.583376,13.26,15.54,19.98,23.89,26.12
equity_folios_crore,21.0,13.880476,3.207799,9.28,10.88,13.99,16.72,18.28
debt_folios_crore,21.0,2.776190,0.640308,1.86,2.18,2.80,3.34,3.66
hybrid_folios_crore,21.0,1.190000,0.274864,0.80,0.93,1.20,1.43,1.57
others_folios_crore,21.0,1.983333,0.459079,1.33,1.55,2.00,2.39,2.61



Rows where total differs from category sum: 11


In [5]:
# Calculate the sum of all folio categories
folio_df["category_sum_crore"] = (
    folio_df["equity_folios_crore"]
    + folio_df["debt_folios_crore"]
    + folio_df["hybrid_folios_crore"]
    + folio_df["others_folios_crore"]
)

# Calculate the difference between reported total and category sum
folio_df["difference_crore"] = (
    folio_df["total_folios_crore"]
    - folio_df["category_sum_crore"]
)

# Display rows where the difference exceeds rounding tolerance
mismatch_df = folio_df[
    folio_df["difference_crore"].abs() > 0.02
]

print("Number of mismatched rows:", len(mismatch_df))

display(
    mismatch_df[
        [
            "month",
            "total_folios_crore",
            "equity_folios_crore",
            "debt_folios_crore",
            "hybrid_folios_crore",
            "others_folios_crore",
            "category_sum_crore",
            "difference_crore",
        ]
    ]
)

Number of mismatched rows: 0


,month,total_folios_crore,equity_folios_crore,debt_folios_crore,hybrid_folios_crore,others_folios_crore,category_sum_crore,difference_crore


In [6]:
# Recalculate the category sum
category_sum = (
    folio_df["equity_folios_crore"]
    + folio_df["debt_folios_crore"]
    + folio_df["hybrid_folios_crore"]
    + folio_df["others_folios_crore"]
)

difference = folio_df["total_folios_crore"] - category_sum

print("Rows with any difference:", (difference != 0).sum())
print("Rows differing by more than 0.02:", (difference.abs() > 0.02).sum())
print("Largest absolute difference:", difference.abs().max())

print("\nAll differences:")
display(pd.DataFrame({
    "month": folio_df["month"],
    "reported_total": folio_df["total_folios_crore"],
    "category_sum": category_sum,
    "difference": difference
}))

Rows with any difference: 14
Rows differing by more than 0.02: 0
Largest absolute difference: 0.010000000000005116

All differences:


,month,reported_total,category_sum,difference
0,2022-01,13.26,13.27,-1.000000e-02
1,2022-04,13.91,13.91,0.000000e+00
2,2022-07,13.85,13.84,1.000000e-02
3,2022-10,14.12,14.12,-1.776357e-15
4,2023-01,14.81,14.81,0.000000e+00
5,2023-04,15.54,15.54,-1.776357e-15
6,2023-07,16.28,16.29,-1.000000e-02
7,2023-10,16.72,16.71,1.000000e-02
8,2024-01,17.78,17.79,-1.000000e-02
9,2024-04,18.85,18.86,-1.000000e-02


In [7]:
# Create a copy of the original dataset
folio_clean = folio_df[
    [
        "month",
        "total_folios_crore",
        "equity_folios_crore",
        "debt_folios_crore",
        "hybrid_folios_crore",
        "others_folios_crore",
    ]
].copy()

# Convert month to datetime
folio_clean["month"] = pd.to_datetime(
    folio_clean["month"],
    format="%Y-%m",
    errors="coerce"
)

# Sort chronologically
folio_clean = folio_clean.sort_values(
    by="month"
).reset_index(drop=True)

# Validate the cleaned dataset
print("Cleaned dataset shape:", folio_clean.shape)

print("\nMissing values:")
print(folio_clean.isnull().sum())

print("\nDuplicate rows:", folio_clean.duplicated().sum())

print(
    "\nDuplicate months:",
    folio_clean["month"].duplicated().sum()
)

print(
    "\nInvalid months:",
    folio_clean["month"].isnull().sum()
)

print("\nCleaned preview:")
display(folio_clean.head())

Cleaned dataset shape: (21, 6)

Missing values:
month                  0
total_folios_crore     0
equity_folios_crore    0
debt_folios_crore      0
hybrid_folios_crore    0
others_folios_crore    0
dtype: int64

Duplicate rows: 0

Duplicate months: 0

Invalid months: 0

Cleaned preview:


,month,total_folios_crore,equity_folios_crore,debt_folios_crore,hybrid_folios_crore,others_folios_crore
0,2022-01-01,13.26,9.28,1.86,0.80,1.33
1,2022-04-01,13.91,9.74,1.95,0.83,1.39
2,2022-07-01,13.85,9.69,1.94,0.83,1.38
3,2022-10-01,14.12,9.88,1.98,0.85,1.41
4,2023-01-01,14.81,10.37,2.07,0.89,1.48


In [8]:
# Define the output path
output_file = (
    project_path
    / "data"
    / "processed"
    / "cleaned_industry_folio_count.csv"
)

# Export the cleaned dataset
folio_clean.to_csv(output_file, index=False)

print("Dataset exported successfully!")
print("File location:", output_file)
print("Exported shape:", folio_clean.shape)

Dataset exported successfully!
File location: d:\Mutual-Fund-Analytics\data\processed\cleaned_industry_folio_count.csv
Exported shape: (21, 6)


In [9]:
# Reload the exported CSV
folio_verified = pd.read_csv(output_file)

print("File exists:", output_file.exists())
print("Reloaded shape:", folio_verified.shape)
print("Columns:", folio_verified.columns.tolist())

display(folio_verified.head())

File exists: True
Reloaded shape: (21, 6)
Columns: ['month', 'total_folios_crore', 'equity_folios_crore', 'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore']


,month,total_folios_crore,equity_folios_crore,debt_folios_crore,hybrid_folios_crore,others_folios_crore
0,2022-01-01,13.26,9.28,1.86,0.80,1.33
1,2022-04-01,13.91,9.74,1.95,0.83,1.39
2,2022-07-01,13.85,9.69,1.94,0.83,1.38
3,2022-10-01,14.12,9.88,1.98,0.85,1.41
4,2023-01-01,14.81,10.37,2.07,0.89,1.48


# Dataset 6 — Industry Folio Count: Cleaning Summary

## Dataset Overview
- **Source file:** `06_industry_folio_count.csv`
- **Output file:** `data/processed/cleaned_industry_folio_count.csv`
- **Final shape:** 21 rows × 6 columns
- **Date range:** January 2022 – December 2025

## Data Quality Checks
- Checked for missing values.
- Checked for duplicate rows and duplicate months.
- Validated month values and converted them to datetime.
- Sorted records chronologically.
- Compared reported total folios against the sum of category folios.

## Findings
- No missing values, duplicate rows, duplicate months, or invalid dates were found.
- Small differences of up to 0.01 crore were observed between reported totals and category sums, consistent with rounding.
- Original folio values were preserved without modification.

## Output
The cleaned dataset was exported to `data/processed/cleaned_industry_folio_count.csv` and verified by reloading the CSV.

## Status
**Cleaning and export verification completed.**